# Databricks: Authenticate and Run a Query

This notebook outlines how to authenticate with Databricks and pull data by running a SQL query. It works **locally** (using env vars and the SQL connector) and **on Databricks** (using Spark).

## 1. Authentication

### Option A: CLI (one-time setup)

From a terminal:

```bash
databricks configure --token
```

You’ll be prompted for:
- **Host**: `https://<your-workspace>.cloud.databricks.com`
- **Token**: Create under **User Settings → Developer → Access tokens**

### Option B: Environment variables (for Python / scripts)

Set these (e.g. in `.env` or `tmp.env`):

- **`DATABRICKS_HOST`** — workspace URL, e.g. `https://<workspace>.cloud.databricks.com`
- **`DATABRICKS_TOKEN`** — personal access token
- **`DATABRICKS_HTTP_PATH`** — SQL warehouse HTTP path (for running SQL from local). Find it in the Databricks UI: **SQL Warehouses → your warehouse → Connection details → HTTP path**

In [ ]:
# Load env vars (e.g. from .env or tmp.env)
import os
from dotenv import load_dotenv

load_dotenv()  # or load_dotenv("tmp.env") if using that file

host = os.getenv("DATABRICKS_HOST")
token = os.getenv("DATABRICKS_TOKEN")
http_path = os.getenv("DATABRICKS_HTTP_PATH")

# Check that required vars are set (host and token are required; http_path only for SQL connector)
assert host, "Set DATABRICKS_HOST in your environment or .env"
assert token, "Set DATABRICKS_TOKEN in your environment or .env"
print(f"Host set: {bool(host)}")
print(f"Token set: {bool(token)}")
print(f"HTTP path set: {bool(http_path)} (required for running SQL from local)")

## 2. Pull a query

**On Databricks:** Use Spark (e.g. `spark.sql("SELECT ...")`).  
**Locally:** Use the Databricks SQL connector and a SQL warehouse (requires `DATABRICKS_HTTP_PATH`).

Below we branch: on Databricks we use Spark; locally we use the SQL connector.

In [ ]:
# Install the SQL connector when running locally (skip on Databricks)
def _is_databricks():
    return "DATABRICKS_RUNTIME_VERSION" in os.environ

if not _is_databricks():
    %pip install -q databricks-sql-connector
    # Restart the kernel after pip if you see import errors, or re-run from here

In [ ]:
from auto_reporting.example_notebooks.utils import is_databricks_runtime

# Example query — replace with your catalog.schema.table and query
EXAMPLE_QUERY = """
SELECT 1 AS id, 'example' AS name
UNION ALL
SELECT 2, 'row two'
"""

if is_databricks_runtime():
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
    df = spark.sql(EXAMPLE_QUERY)
    df.show()
else:
    # Local: use Databricks SQL connector (requires DATABRICKS_HTTP_PATH)
    assert http_path, "Set DATABRICKS_HTTP_PATH (SQL warehouse HTTP path) to run queries locally"
    from databricks import sql as databricks_sql

    with databricks_sql.connect(
        server_hostname=host.replace("https://", "").strip("/"),
        http_path=http_path,
        access_token=token,
    ) as conn:
        cursor = conn.cursor()
        cursor.execute(EXAMPLE_QUERY)
        rows = cursor.fetchall()
        column_names = [desc[0] for desc in cursor.description]
    # Display as a simple table
    import pandas as pd
    df = pd.DataFrame(rows, columns=column_names)
    display(df)

### Running a query from the CLI

If the Databricks CLI is configured (`databricks configure --token`) and you have a **SQL warehouse**, you can run one-off queries from a terminal:

```bash
# List warehouses to get the warehouse ID
databricks sql warehouses list

# Run a query (replace <warehouse-id> with the id from the list)
databricks sql execute --warehouse-id <warehouse-id> --query "SELECT * FROM your_catalog.your_schema.your_table LIMIT 10"
```